# 특수 타입 파싱하기 - datetime과 Enum을 Pydantic으로 다뤄요

## 학습 목표

이 노트북을 마치면 다음을 할 수 있어요.

- Pydantic 모델에 `datetime` 필드를 선언해 LLM 출력을 날짜/시간 객체로 자동 변환할 수 있어요
- Python `Enum`(열거형)을 Pydantic 필드로 사용해 LLM 출력을 제한된 선택지 중 하나로 파싱할 수 있어요
- 날짜 연산(D-Day 계산 등)과 카테고리 분류를 LLM 파이프라인에 통합할 수 있어요

## 사전 지식

- **datetime**: Python 표준 라이브러리의 날짜/시간 객체예요. 문자열 `"2024-03-15"`을 파싱하면 날짜 계산, 포맷 변환 등이 가능해져요
- **Enum(열거형)**: 미리 정해진 상수 집합을 표현하는 Python 타입이에요. `Priority.HIGH`, `Sentiment.POSITIVE`처럼 유효한 값만 허용해서 잘못된 분류를 방지해요

## LangChain 버전 참고

LangChain 1.0부터 `DatetimeOutputParser`와 `EnumOutputParser`가 제거됐어요. 대신 `PydanticOutputParser`에 `datetime`과 `Enum` 필드를 조합하는 방식을 사용해요.

| 구버전 | 현재 방식 |
|--------|-----------|
| `DatetimeOutputParser` | `PydanticOutputParser` + `datetime` 필드 |
| `EnumOutputParser` | `PydanticOutputParser` + `Enum` 필드 |

## 파이프라인 흐름

```mermaid
flowchart LR
    A["PromptTemplate"] -->|"형식 지침 포함"| B["ChatOpenAI"]
    B -->|"JSON 텍스트"| C["PydanticOutputParser"]
    C -->|"datetime 필드"| D["datetime 객체<br/>날짜 계산·포맷 변환 가능"]
    C -->|"Enum 필드"| E["Enum 값<br/>정해진 선택지만 허용"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C process
    class D,E output
```

In [1]:
from dotenv import load_dotenv

load_dotenv()


True

# 1. datetime 타입 파싱

Pydantic 모델에 `datetime` 필드를 선언하면 LLM이 반환한 날짜 문자열을 자동으로 Python `datetime` 객체로 변환해줘요.

## 지원하는 날짜 형식

Pydantic은 다양한 날짜 형식을 자동으로 인식해요.

| 형식 | 예시 |
|------|------|
| ISO 8601 기본 | `2024-03-15` |
| 날짜 + 시간 | `2024-03-15T14:30:00` |
| 타임존 포함 | `2024-03-15T14:30:00+09:00` |

> 🎯 **강의 포인트**: Pydantic의 `datetime` 필드는 문자열을 받아서 자동으로 `datetime` 객체로 변환해줘요. 변환 후에는 날짜 계산(`timedelta`), 포맷 변환(`strftime`) 등 Python datetime의 모든 기능을 그대로 사용할 수 있어요.

> 💡 **실무 팁**: 프롬프트에 `"YYYY-MM-DD 형식"`처럼 형식을 명시하면 LLM이 일관된 날짜 형식을 반환해서 파싱 실패를 줄일 수 있어요.

## 실용 예제 1: 역사적 사건 날짜 추출

역사적 사건의 날짜를 추출하여 datetime 객체로 변환하는 시나리오입니다.


In [2]:
from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1단계: Pydantic 모델 정의 (datetime 필드 포함)
class DateInfo(BaseModel):
    """날짜 정보"""
    date: datetime = Field(description="날짜 (YYYY-MM-DD 형식)")

# 2단계: PydanticOutputParser 생성
parser = PydanticOutputParser(pydantic_object=DateInfo)

# 3단계: 프롬프트 템플릿 생성
template = """다음 질문에 대한 답변을 날짜 형식으로 제공하세요.

질문: {question}

{format_instructions}

답변:"""

prompt = PromptTemplate.from_template(
    template,
    partial_variables={"format_instructions": parser.get_format_instructions()},
)

# 4단계: 체인 구성
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)
chain = prompt | model | parser

# 5단계: 실행
result = chain.invoke({"question": "대한민국이 독립한 날짜"})


In [3]:
# ---------------------------------------------------
# 체인 실행 결과 출력 - datetime 객체 활용
# ---------------------------------------------------

print("=" * 60)
print("📅 추출된 날짜 정보")
print("=" * 60)
# result.date: Pydantic이 문자열을 datetime 객체로 자동 변환
print(f"타입: {type(result)}")
print(f"날짜 객체: {result.date}")
# 🎯 강의 포인트: datetime 객체로 변환됐기 때문에 strftime, 날짜 연산 등이 바로 가능
print(f"포맷 변환: {result.date.strftime('%Y년 %m월 %d일')}")
# %A: 요일 이름 (영문)
print(f"요일: {result.date.strftime('%A')}")

📅 추출된 날짜 정보
타입: <class '__main__.DateInfo'>
날짜 객체: 1945-08-15 00:00:00
포맷 변환: 1945년 08월 15일
요일: Wednesday


## 실용 예제 2: 이벤트 일정 관리

이벤트 정보에서 날짜를 추출하고 D-Day를 계산하는 시나리오입니다.


In [4]:
# ---------------------------------------------------
# 이벤트 일정 관리 - datetime으로 D-Day 계산
# ---------------------------------------------------

from datetime import datetime

# Pydantic 모델 정의 (날짜와 시간 포함)
class EventDate(BaseModel):
    """이벤트 날짜 및 시간"""
    # YYYY-MM-DDTHH:MM 형식: ISO 8601 날짜+시간 형식 (T가 날짜와 시간을 구분)
    event_datetime: datetime = Field(description="이벤트 시작 날짜와 시간 (YYYY-MM-DDTHH:MM 형식)")

# PydanticOutputParser 생성
event_parser = PydanticOutputParser(pydantic_object=EventDate)

# 프롬프트 템플릿
event_prompt = PromptTemplate.from_template(
    """다음 이벤트 정보에서 시작 날짜와 시간을 추출하세요.

이벤트 정보:
{event_info}

{format_instructions}"""
)

event_prompt = event_prompt.partial(
    format_instructions=event_parser.get_format_instructions()
)

# 체인 구성
event_chain = event_prompt | model | event_parser

# 실행
event_info = """
2025년 봄 개발자 컨퍼런스가 3월 20일 오후 2시에 개최됩니다.
장소는 서울 강남구 코엑스이며, 사전 등록이 필요합니다.
"""

result = event_chain.invoke({"event_info": event_info})
event_date = result.event_datetime

# D-Day 계산 - datetime 객체이므로 뺄셈 연산 가능
now = datetime.now()
days_until = (event_date - now).days

print("\n" + "=" * 60)
print("📆 이벤트 일정 정보")
print("=" * 60)
print(f"이벤트 날짜: {event_date.strftime('%Y년 %m월 %d일 %H시 %M분')}")
print(f"현재 시각: {now.strftime('%Y년 %m월 %d일 %H시 %M분')}")
print(f"D-Day: {days_until}일 남음" if days_until > 0 else "이벤트가 종료되었습니다.")


📆 이벤트 일정 정보
이벤트 날짜: 2025년 03월 20일 14시 00분
현재 시각: 2026년 03월 25일 13시 29분
이벤트가 종료되었습니다.


# 2. Enum 타입 파싱

Python의 `Enum`(열거형)을 Pydantic 필드로 사용하면 LLM 출력을 **미리 정해진 값 중 하나**로 제한할 수 있어요. 유효하지 않은 값은 자동으로 거부돼요.

## 언제 유용할까요?

- **카테고리 분류**: 날씨(맑음/흐림/비/눈), 감정(긍정/중립/부정)
- **우선순위 설정**: 높음/보통/낮음
- **상태 관리**: 승인/대기/거부

> 🔑 **핵심 개념**: `Enum`은 LLM이 임의의 텍스트를 반환하지 못하도록 막는 "안전장치"예요. 허용된 값 이외를 반환하면 `ValidationError`가 자동으로 발생합니다.

> ⚠️ **자주 하는 실수**: LLM이 Enum에 없는 값을 반환하면 `ValidationError`가 발생해요. Enum 값 설명을 `Field(description="...")` 에 명확히 적어서 LLM이 올바른 선택지를 고르도록 유도하세요.

## 실용 예제 1: 날씨 상태 분류

날씨 설명을 간단한 카테고리로 분류하는 시나리오입니다.


In [5]:
from enum import Enum
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1단계: Enum 클래스 정의
class WeatherType(Enum):
    SUNNY = "맑음"
    CLOUDY = "흐림"
    RAINY = "비"
    SNOWY = "눈"

# 2단계: Pydantic 모델 정의 (Enum 필드 포함)
class WeatherClassification(BaseModel):
    """날씨 분류"""
    weather: WeatherType = Field(description="날씨 카테고리 (맑음, 흐림, 비, 눈 중 하나)")

# 3단계: PydanticOutputParser 생성
weather_parser = PydanticOutputParser(pydantic_object=WeatherClassification)

# 4단계: 프롬프트 템플릿
weather_prompt = PromptTemplate.from_template(
    """다음 날씨 설명을 간단한 카테고리로 분류하세요.

날씨 설명: {description}

{format_instructions}"""
).partial(format_instructions=weather_parser.get_format_instructions())

# 5단계: 체인 구성
weather_chain = weather_prompt | model | weather_parser

# 6단계: 실행
result = weather_chain.invoke({"description": "오늘은 구름이 많고 가끔 햇빛이 비칩니다."})

print("\n" + "=" * 60)
print("🌤️ 날씨 분류 결과")
print("=" * 60)
print(f"타입: {type(result)}")
print(f"결과: {result.weather}")
print(f"값: {result.weather.value}")



🌤️ 날씨 분류 결과
타입: <class '__main__.WeatherClassification'>
결과: WeatherType.CLOUDY
값: 흐림


## 실용 예제 2: 태스크 우선순위 설정

프로젝트 태스크의 우선순위를 자동으로 분류하는 시나리오입니다.


In [6]:
# ---------------------------------------------------
# 실용 예제 2: 태스크 우선순위 자동 분류 (Enum)
# ---------------------------------------------------

# Enum 정의 - 허용할 우선순위 값 지정
class Priority(Enum):
    HIGH = "높음"
    MEDIUM = "보통"
    LOW = "낮음"

# Pydantic 모델 정의
class TaskPriority(BaseModel):
    """태스크 우선순위"""
    # Enum 필드: 정의된 값(높음/보통/낮음) 이외의 값은 ValidationError 발생
    priority: Priority = Field(description="우선순위 (높음, 보통, 낮음 중 하나)")

# 파서 생성
priority_parser = PydanticOutputParser(pydantic_object=TaskPriority)

# 프롬프트
priority_prompt = PromptTemplate.from_template(
    """다음 태스크의 우선순위를 평가하세요.

태스크: {task}
마감일: {deadline}
영향도: {impact}

{format_instructions}"""
).partial(format_instructions=priority_parser.get_format_instructions())

# 체인 구성
priority_chain = priority_prompt | model | priority_parser

# 여러 태스크에 대해 실행
tasks = [
    {"task": "보안 취약점 수정", "deadline": "내일까지", "impact": "서비스 전체에 영향"},
    {"task": "UI 색상 개선", "deadline": "다음 주", "impact": "일부 사용자 경험 향상"},
    {"task": "코드 리팩토링", "deadline": "이번 달", "impact": "유지보수성 향상"},
]

print("=" * 60)
print("📋 태스크 우선순위 자동 분류")
print("=" * 60)
for task_info in tasks:
    result = priority_chain.invoke(task_info)
    priority = result.priority
    # 💡 팁: Enum 값 비교는 == 연산자로, 실제 문자열 값은 .value로 꺼냄
    emoji = "🔴" if priority == Priority.HIGH else "🟡" if priority == Priority.MEDIUM else "🟢"
    print(f"{emoji} {task_info['task']}: {priority.value}")

📋 태스크 우선순위 자동 분류
🔴 보안 취약점 수정: 높음
🟡 UI 색상 개선: 보통
🔴 코드 리팩토링: 높음


## 실용 예제 3: 고객 피드백 감정 분석

고객 피드백의 감정을 분류하는 시나리오입니다.


In [7]:
# ---------------------------------------------------
# 실용 예제 3: 고객 피드백 감정 분석 (Enum)
# ---------------------------------------------------

from langchain_core.prompts import ChatPromptTemplate

# Enum 정의 - 감정 분류 선택지
class Sentiment(Enum):
    POSITIVE = "긍정"
    NEUTRAL = "중립"
    NEGATIVE = "부정"

# Pydantic 모델 정의
class SentimentAnalysis(BaseModel):
    """감정 분석 결과"""
    sentiment: Sentiment = Field(description="감정 분류 (긍정, 중립, 부정 중 하나)")

# 파서 생성
sentiment_parser = PydanticOutputParser(pydantic_object=SentimentAnalysis)

# 프롬프트
sentiment_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 고객 피드백 분석 전문가입니다."),
    ("user", """다음 고객 피드백의 감정을 분석하세요.

피드백: {feedback}

{format_instructions}""")
]).partial(format_instructions=sentiment_parser.get_format_instructions())

# 체인 구성
sentiment_chain = sentiment_prompt | model | sentiment_parser

# 여러 피드백 분석
feedbacks = [
    "제품이 정말 훌륭해요! 배송도 빠르고 품질도 최고입니다.",
    "배송은 빨랐지만 제품이 설명과 달라서 실망했습니다.",
    "보통입니다. 가격 대비 괜찮은 것 같아요.",
]

print("\n" + "=" * 60)
print("💬 고객 피드백 감정 분석")
print("=" * 60)
for i, feedback in enumerate(feedbacks, 1):
    result = sentiment_chain.invoke({"feedback": feedback})
    sentiment = result.sentiment
    # .value: Enum 멤버의 실제 값 반환 (예: Sentiment.POSITIVE.value → "긍정")
    emoji = "😊" if sentiment == Sentiment.POSITIVE else "😐" if sentiment == Sentiment.NEUTRAL else "😞"
    print(f"\n{i}. {feedback[:30]}...")
    print(f"   {emoji} 감정: {sentiment.value}")


💬 고객 피드백 감정 분석

1. 제품이 정말 훌륭해요! 배송도 빠르고 품질도 최고입니다...
   😊 감정: 긍정

2. 배송은 빨랐지만 제품이 설명과 달라서 실망했습니다....
   😞 감정: 부정

3. 보통입니다. 가격 대비 괜찮은 것 같아요....
   😐 감정: 중립


## 핵심 요약

| 타입 | 사용 방법 | 주요 활용 |
|------|-----------|-----------|
| `datetime` | Pydantic 필드 타입으로 선언 | 날짜 추출, D-Day 계산, 이벤트 일정 |
| `Enum` | `class MyEnum(Enum)` + Pydantic 필드 | 카테고리 분류, 우선순위, 감정 분석 |

### 두 타입을 하나의 모델에 조합할 수 있어요

```mermaid
flowchart LR
    A["LLM 응답<br/>(JSON 텍스트)"] --> B["PydanticOutputParser"]
    B --> C["Pydantic 모델<br/>event_date: datetime<br/>priority: PriorityEnum<br/>title: str"]
    C --> D["event.event_date.strftime(...)"]
    C --> E["event.priority == Priority.HIGH"]
    C --> F["event.title"]

    classDef input fill:#d4edda,stroke:#28a745,color:#155724
    classDef process fill:#cce5ff,stroke:#007bff,color:#004085
    classDef output fill:#fff3cd,stroke:#ffc107,color:#856404

    class A input
    class B,C process
    class D,E,F output
```

## 다음 노트북 예고

**노트북 05 - PandasDataFrameOutputParser** 에서는 기존 Pandas DataFrame(데이터프레임)을 LLM에 연결해 자연어로 데이터를 조회하는 방법을 배워요. SQL 없이 한국어로 데이터를 분석할 수 있어요.

## 연습 문제

다음 연습 문제를 통해 Pydantic 기반 `datetime`과 `Enum` 타입 파싱을 직접 실습해봅시다.

### 연습 1: IT 역사 타임라인 (datetime 파싱)

`PydanticOutputParser`와 `datetime` 필드를 사용하여 IT 역사의 주요 사건 날짜를 추출하는 체인을 만들어보세요.

**요구사항:**
- Pydantic 모델 `HistoricalEvent`를 정의하세요:
  - `event_name` (str): 사건 이름
  - `event_date` (datetime): 사건 날짜
  - `description` (str): 사건 설명 (한 문장)
- 3가지 질문에 대해 각각 체인을 실행하세요:
  1. "최초의 iPhone이 출시된 날짜"
  2. "월드 와이드 웹(WWW)이 처음 공개된 날짜"
  3. "ChatGPT가 처음 출시된 날짜"
- 각 결과의 날짜를 "YYYY년 MM월 DD일" 형식으로 출력하세요

In [8]:
# ============================================================
# TODO: IT 역사 타임라인 만들기 (datetime 파싱)
# 힌트: HistoricalEvent 모델에 datetime 필드를 포함하고,
#       result.event_date.strftime()으로 날짜를 포맷하세요
# 예상 결과: 각 사건의 날짜가 "YYYY년 MM월 DD일" 형식으로 출력됨
# ============================================================

# 연습 1 풀이

from datetime import datetime
from langchain_openai import ChatOpenAI
from langchain_core.prompts import PromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1단계: Pydantic 모델 정의 (datetime 필드 포함)
class HistoricalEvent(BaseModel):
    """IT 역사적 사건 정보"""
    event_name: str = Field(description="사건 이름")
    # datetime 필드: Pydantic이 YYYY-MM-DD 문자열을 datetime 객체로 자동 변환
    event_date: datetime = Field(description="사건 날짜 (YYYY-MM-DD 형식)")
    description: str = Field(description="사건에 대한 한 문장 설명")

# 2단계: 파서 및 모델 초기화
parser = PydanticOutputParser(pydantic_object=HistoricalEvent)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 3단계: 프롬프트 구성
prompt = PromptTemplate.from_template(
    """다음 IT 역사적 사건에 대한 정보를 제공하세요.

사건: {question}

{format_instructions}"""
)
prompt = prompt.partial(format_instructions=parser.get_format_instructions())

# 4단계: 체인 구성
chain = prompt | model | parser

# 5단계: 3가지 사건에 대해 실행
questions = [
    "최초의 iPhone이 출시된 날짜",
    "월드 와이드 웹(WWW)이 처음 공개된 날짜",
    "ChatGPT가 처음 출시된 날짜",
]

print("=" * 60)
print("IT 역사 타임라인")
print("=" * 60)

for q in questions:
    result = chain.invoke({"question": q})
    # strftime("%Y년 %m월 %d일"): datetime → 한국어 날짜 형식 문자열
    formatted_date = result.event_date.strftime("%Y년 %m월 %d일")
    print(f"\n[{result.event_name}]")
    print(f"  날짜: {formatted_date}")
    print(f"  설명: {result.description}")

IT 역사 타임라인

[최초의 iPhone 출시]
  날짜: 2007년 06월 29일
  설명: 애플이 최초의 iPhone을 출시하여 스마트폰 시장에 혁신을 가져왔다.

[월드 와이드 웹(WWW) 공개]
  날짜: 1991년 08월 06일
  설명: 팀 버너스리(Tim Berners-Lee)가 월드 와이드 웹을 처음으로 공개한 날.

[ChatGPT 출시]
  날짜: 2022년 11월 30일
  설명: OpenAI가 ChatGPT를 처음으로 출시한 날짜.


### 연습 2: 기술 면접 답변 평가기 (Enum 파싱)

`PydanticOutputParser`와 `Enum` 필드를 사용하여 기술 면접 답변의 품질을 평가하는 체인을 만들어보세요.

**요구사항:**
- Enum 클래스 `AnswerQuality`를 정의하세요:
  - `EXCELLENT` = "우수"
  - `GOOD` = "양호"
  - `AVERAGE` = "보통"
  - `POOR` = "미흡"
- Pydantic 모델 `InterviewEvaluation`을 정의하세요:
  - `quality` (AnswerQuality): 답변 품질
  - `feedback` (str): 피드백 코멘트 (한 문장)
- 3가지 면접 답변을 평가하세요:
  1. 질문: "REST API란?", 답변: "REST API는 HTTP 프로토콜을 기반으로 자원을 URI로 표현하고 HTTP 메서드로 CRUD 작업을 수행하는 아키텍처 스타일입니다."
  2. 질문: "Git이란?", 답변: "코드를 저장하는 것입니다."
  3. 질문: "Docker란?", 답변: "Docker는 컨테이너 기술을 사용하여 애플리케이션을 격리된 환경에서 실행할 수 있게 해주는 플랫폼으로, 개발 환경과 운영 환경의 일관성을 보장합니다."

In [9]:
# ============================================================
# TODO: 기술 면접 답변 평가기 만들기 (Enum 파싱)
# 힌트: AnswerQuality Enum과 InterviewEvaluation 모델을 정의하고
#       ChatPromptTemplate으로 시스템 메시지를 포함하세요
# 예상 결과: 각 답변이 우수/양호/보통/미흡 중 하나로 분류되어 출력됨
# ============================================================

# 연습 2 풀이

from enum import Enum
from langchain_openai import ChatOpenAI
from langchain_core.prompts import ChatPromptTemplate
from langchain_core.output_parsers import PydanticOutputParser
from pydantic import BaseModel, Field

# 1단계: Enum 정의 - 평가 등급 선택지
class AnswerQuality(Enum):
    EXCELLENT = "우수"
    GOOD = "양호"
    AVERAGE = "보통"
    POOR = "미흡"

# 2단계: Pydantic 모델 정의
class InterviewEvaluation(BaseModel):
    """면접 답변 평가 결과"""
    # Enum 필드: 4가지 값 중 하나만 허용
    quality: AnswerQuality = Field(description="답변 품질 (우수, 양호, 보통, 미흡 중 하나)")
    feedback: str = Field(description="피드백 코멘트 (한 문장)")

# 3단계: 파서 및 모델 초기화
eval_parser = PydanticOutputParser(pydantic_object=InterviewEvaluation)
model = ChatOpenAI(model="gpt-4o-mini", temperature=0)

# 4단계: 프롬프트 구성
eval_prompt = ChatPromptTemplate.from_messages([
    ("system", "당신은 기술 면접관입니다. 답변의 정확성과 깊이를 평가하세요."),
    ("user", """다음 면접 답변을 평가하세요.

질문: {question}
답변: {answer}

{format_instructions}""")
]).partial(format_instructions=eval_parser.get_format_instructions())

# 5단계: 체인 구성
eval_chain = eval_prompt | model | eval_parser

# 6단계: 3가지 답변 평가
interview_data = [
    {
        "question": "REST API란?",
        "answer": "REST API는 HTTP 프로토콜을 기반으로 자원을 URI로 표현하고 HTTP 메서드로 CRUD 작업을 수행하는 아키텍처 스타일입니다."
    },
    {
        "question": "Git이란?",
        "answer": "코드를 저장하는 것입니다."
    },
    {
        "question": "Docker란?",
        "answer": "Docker는 컨테이너 기술을 사용하여 애플리케이션을 격리된 환경에서 실행할 수 있게 해주는 플랫폼으로, 개발 환경과 운영 환경의 일관성을 보장합니다."
    },
]

print("=" * 60)
print("기술 면접 답변 평가 결과")
print("=" * 60)

for i, data in enumerate(interview_data, 1):
    result = eval_chain.invoke(data)
    # .value: Enum 멤버의 실제 값 (예: AnswerQuality.EXCELLENT.value → "우수")
    marker = {"우수": "A+", "양호": "B+", "보통": "C", "미흡": "D"}
    grade = marker.get(result.quality.value, "?")
    print(f"\n{i}. Q: {data['question']}")
    print(f"   A: {data['answer'][:40]}...")
    print(f"   평가: [{grade}] {result.quality.value}")
    print(f"   피드백: {result.feedback}")

기술 면접 답변 평가 결과

1. Q: REST API란?
   A: REST API는 HTTP 프로토콜을 기반으로 자원을 URI로 표현하고 ...
   평가: [B+] 양호
   피드백: REST API의 기본 개념을 잘 설명했으나, 상태 비저장성(stateless) 및 클라이언트-서버 구조와 같은 추가적인 특징을 언급하면 더 좋을 것입니다.

2. Q: Git이란?
   A: 코드를 저장하는 것입니다....
   평가: [D] 미흡
   피드백: Git의 정의가 너무 간단하며, 버전 관리 시스템으로서의 기능과 특징에 대한 설명이 부족합니다.

3. Q: Docker란?
   A: Docker는 컨테이너 기술을 사용하여 애플리케이션을 격리된 환경에서 실...
   평가: [B+] 양호
   피드백: Docker의 기본 개념을 잘 설명했으나, 컨테이너의 이점이나 작동 방식에 대한 추가 설명이 있으면 더 좋았을 것입니다.
